# NFI Woodland GB — Exploratory Data Analysis

**Rules to re-read before proceeding:** 1 (success criteria before tuning), 2 (EDA before modelling), 3 (correct CRS), 4 (document parameters inline)

**Data source:** NFI Woodland GB 2023 release, downloaded from Forestry Commission ArcGIS Hub
  https://data-forestry.opendata.arcgis.com

**Access date:** <!-- fill in when you download -->  
**File:** `data/raw/NFI_Woodland_GB_<version>.shp` (gitignored — hundreds of MB)

## Goal
Understand the raw NFI data before any clustering:
1. Polygon count and file size
2. IFT_IOA field distribution (which types exist, how many of each)
3. Area statistics (min, max, median — effect of 10ha filter)
4. Geographic distribution (bounding box, density heatmap)
5. Null / data quality check
6. How many polygons survive the conifer + 10ha filter?
7. File size estimate for the filtered + dissolved cluster output

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

# Paths — relative to project root
PROJECT_ROOT = Path('../../')  # analysis/notebooks/ → project root
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR   = PROJECT_ROOT / 'data' / 'output'

print('Project root:', PROJECT_ROOT.resolve())
print('Raw data dir:', RAW_DATA_DIR.resolve())
print('Files in raw:', list(RAW_DATA_DIR.iterdir()) if RAW_DATA_DIR.exists() else 'NOT FOUND')

## Download instructions

If `data/raw/` is empty, download the NFI shapefile:

1. Go to: https://data-forestry.opendata.arcgis.com
2. Search: **"NFI Woodland GB"**
3. Download the **Shapefile** format (the full GB dataset, not just England)
4. Unzip into `data/raw/`
5. Note the release year/version and today's date below

Update the cell below with the actual filename once downloaded.

In [ ]:
# UPDATE THIS when you know the actual filename
NFI_FILE = RAW_DATA_DIR / 'NFI_Woodland_GB.shp'  # adjust if filename differs

# Source citation — fill in after download
NFI_SOURCE = {
    'name': 'NFI Woodland GB',
    'release': '2023',       # <- update with actual release year
    'provider': 'Forestry Commission, via ArcGIS Hub',
    'url': 'https://data-forestry.opendata.arcgis.com',
    'access_date': '2026-05-23',  # <- update with actual download date
    'licence': 'Open Government Licence v3.0'
}
print('Will load:', NFI_FILE)
print('File exists:', NFI_FILE.exists())

## 1. Load data

In [ ]:
print('Loading NFI shapefile — this may take 1-2 minutes for the full GB dataset...')
gdf = gpd.read_file(NFI_FILE)
print(f'Loaded {len(gdf):,} polygons')
print(f'CRS: {gdf.crs}')  # should be EPSG:27700 (OSGB36 / British National Grid)
print(f'Columns: {list(gdf.columns)}')

In [ ]:
# Assert CRS is OSGB36 — Rule 3
assert gdf.crs.to_epsg() == 27700, f'Expected EPSG:27700, got {gdf.crs}. Reproject first!'
print('CRS check passed: EPSG:27700 (metres) — correct for area calculations')

## 2. Data overview

In [ ]:
gdf.head()

In [ ]:
gdf.dtypes

## 3. Null / data quality check

In [ ]:
null_counts = gdf.isnull().sum()
print('Null values per column:')
print(null_counts[null_counts > 0].to_string())
print(f'\nGeometry null: {gdf.geometry.isnull().sum()}')
print(f'Geometry invalid: {(~gdf.geometry.is_valid).sum()}')

## 4. IFT_IOA distribution

This is the key classification field. We want:
- `Conifer` — pure or predominantly conifer
- `Mixed mainly conifer` — >50% conifer by area

In [ ]:
# Check the actual field name — NFI uses IFT_IOA but confirm
type_field = 'IFT_IOA'  # adjust if the column name differs
if type_field not in gdf.columns:
    print('WARNING: IFT_IOA not found. Available columns:', list(gdf.columns))
else:
    counts = gdf[type_field].value_counts(dropna=False)
    print(f'IFT_IOA value counts (total {len(gdf):,}):')
    print(counts.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
counts.plot(kind='bar', ax=ax, color='forestgreen')
ax.set_title('NFI polygon count by IFT_IOA type')
ax.set_xlabel('IFT_IOA')
ax.set_ylabel('Polygon count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Area statistics

In [ ]:
# Area in hectares (EPSG:27700 is in metres, so divide m² by 10000)
gdf['area_ha'] = gdf.geometry.area / 10_000

print('Area statistics (all polygons, hectares):')
print(gdf['area_ha'].describe().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Full distribution
gdf['area_ha'].clip(upper=gdf['area_ha'].quantile(0.99)).hist(
    bins=60, ax=axes[0], color='forestgreen', edgecolor='white'
)
axes[0].set_title('Polygon area distribution (clipped at 99th percentile)')
axes[0].set_xlabel('Area (ha)')
axes[0].set_ylabel('Count')

# Zoom: 0-50ha to see where the 10ha threshold sits
gdf['area_ha'].clip(upper=50).hist(bins=50, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].axvline(10, color='red', linestyle='--', label='10ha threshold')
axes[1].set_title('Polygon area 0–50ha (showing 10ha filter)')
axes[1].set_xlabel('Area (ha)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nPolygons >= 10ha: {(gdf["area_ha"] >= 10).sum():,} ({(gdf["area_ha"] >= 10).mean():.1%})')
print(f'Polygons < 10ha:  {(gdf["area_ha"] < 10).sum():,} ({(gdf["area_ha"] < 10).mean():.1%})')

## 6. Apply conifer filter + 10ha threshold

This is the input to DBSCAN — document the decision.

In [ ]:
# Filter to conifer types — Rule 4: document why
CONIFER_TYPES = [
    'Conifer',             # pure or predominantly conifer
    'Mixed mainly conifer' # >50% conifer by area
]
# 10ha minimum: small isolated patches are irrelevant as holiday destinations
AREA_MIN_HA = 10

conifer = gdf[
    gdf[type_field].isin(CONIFER_TYPES) &
    (gdf['area_ha'] >= AREA_MIN_HA)
].copy()

print(f'All NFI polygons:             {len(gdf):>8,}')
print(f'Conifer types only:           {len(gdf[gdf[type_field].isin(CONIFER_TYPES)]):>8,}')
print(f'Conifer + >= {AREA_MIN_HA}ha:          {len(conifer):>8,}')
print(f'\nConifer total area: {conifer["area_ha"].sum():,.0f} ha')
print(f'Conifer type breakdown:')
print(conifer[type_field].value_counts().to_string())

## 7. Geographic distribution

In [ ]:
# Bounding box of filtered dataset
bounds = conifer.total_bounds  # minx, miny, maxx, maxy in EPSG:27700 (metres)
print(f'Bounding box (EPSG:27700, metres):')
print(f'  Easting:   {bounds[0]:,.0f} – {bounds[2]:,.0f}')
print(f'  Northing:  {bounds[1]:,.0f} – {bounds[3]:,.0f}')

# Centroid scatter — Rule 8: visual sanity check
centroids = conifer.copy()
centroids['x'] = conifer.geometry.centroid.x
centroids['y'] = conifer.geometry.centroid.y

fig, ax = plt.subplots(figsize=(8, 12))
scatter = ax.scatter(
    centroids['x'], centroids['y'],
    c=centroids['area_ha'],
    cmap='YlGn',
    s=2,
    alpha=0.6,
    norm=mcolors.LogNorm()
)
plt.colorbar(scatter, ax=ax, label='Area (ha, log scale)')
ax.set_title(f'Conifer NFI centroids (n={len(conifer):,})')
ax.set_xlabel('Easting (m, EPSG:27700)')
ax.set_ylabel('Northing (m, EPSG:27700)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print('\nExpected: dense cluster in Scottish Highlands, Kielder, Thetford, Wales, Kielder')

## 8. File size estimate for output

In [ ]:
import tempfile, os

# Save a 1% sample to estimate GeoJSON size
sample = conifer.sample(frac=0.01, random_state=42)
with tempfile.NamedTemporaryFile(suffix='.geojson', delete=False) as f:
    sample_path = f.name

# Reproject to WGS84 for the estimate (output will be WGS84 for the web app)
sample_wgs = sample.to_crs(4326)
sample_wgs.to_file(sample_path, driver='GeoJSON')
sample_size_mb = os.path.getsize(sample_path) / 1e6
os.unlink(sample_path)

estimated_full_mb = sample_size_mb * 100  # 1% sample → extrapolate
print(f'1% sample ({len(sample)} polygons): {sample_size_mb:.2f} MB')
print(f'Estimated full filtered GeoJSON: ~{estimated_full_mb:.0f} MB')
print()
print('After DBSCAN dissolve (~100–300 cluster polygons):')
print('  Estimated: <5MB (dissolved polygons are few; vertices simplified during dissolve)')
print('  Decision threshold: >20MB → PMTiles; <20MB → serve GeoJSON directly')

## 9. Key numbers summary

Fill this in after running the cells above — this becomes the methodology section.

In [ ]:
print('=== EDA SUMMARY ===')
print(f'Total NFI polygons:           {len(gdf):,}')
print(f'Conifer types + >=10ha:       {len(conifer):,}')
print(f'Reduction:                    {1 - len(conifer)/len(gdf):.1%} of polygons removed by filter')
print(f'Total conifer area:           {conifer["area_ha"].sum():,.0f} ha')
print(f'Median patch area:            {conifer["area_ha"].median():.1f} ha')
print(f'Largest patch:                {conifer["area_ha"].max():.0f} ha')
print()
print('Next step: 02_clustering.ipynb')
print('Set DBSCAN success criteria BEFORE running clustering (Rule 1)')

## Next steps

Before moving to `02_clustering.ipynb`, document your success criteria here (Rule 1):

```
I will accept the clustering result if:
- Thetford emerges as approximately 1 cluster
- Kielder emerges as approximately 1 cluster
- Total clusters: between [X] and [Y]        ← fill in after reviewing centroid plot
- Noise (unclustered) polygons: under 10%
```

**Only then** open the clustering notebook.